In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

print(os.listdir("/kaggle/input/competitions/rsna-pneumonia-detection-challenge"))

df = pd.read_csv("/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv")
patients = df.drop_duplicates("patientId").reset_index(drop=True)
patients.head(10)

train_df, temp_df = train_test_split(
    patients, test_size=0.3, stratify=patients["Target"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.333, stratify=temp_df["Target"], random_state=42
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)
test_df.to_csv("test_split.csv", index=False))

bbox_lookup = {}
for pid, group in df[df["Target"] == 1].groupby("patientId"):
    bbox_lookup[pid] = [
        {
            "x": r.x * 224 / 1024,
            "y": r.y * 224 / 1024,
            "w": r.width * 224 / 1024,
            "h": r.height * 224 / 1024,
        }
        for _, r in group.iterrows()
    ]


token = ""
api = HfApi(token=token)

REPO_ID = "anonymous"
IMG_DIR = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"

api.create_repo(repo_id=REPO_ID, repo_type="dataset", private=True, exist_ok=True)

UPLOAD_ROOT = "/kaggle/working/upload_root"
UPLOAD_IMAGES_DIR = os.path.join(UPLOAD_ROOT, "images")
SHARD_SIZE = 1000

all_files = sorted(os.listdir(IMG_DIR))
for i, fname in enumerate(all_files):
    shard_dir = os.path.join(UPLOAD_IMAGES_DIR, f"shard_{i // SHARD_SIZE:03d}")
    os.makedirs(shard_dir, exist_ok=True)
    dst = os.path.join(shard_dir, fname)
    if not os.path.exists(dst):
        os.symlink(os.path.join(IMG_DIR, fname), dst)

api.upload_large_folder(
    folder_path=UPLOAD_ROOT,
    repo_id=REPO_ID,
    repo_type="dataset",
)
print("DCM folder upload done")

for f, name in [
    ("/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv", "stage_2_train_labels.csv"),
    ("train_split.csv", "train_split.csv"),
    ("val_split.csv", "val_split.csv"),
    ("test_split.csv", "test_split.csv"),
]:
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=name,
        repo_id=REPO_ID,
        repo_type="dataset",
    )
print("CSVs uploaded")
print("Done.")